In [1]:
!pip install youtube-transcript-api yt-dlp openai-whisper -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.2/182.2 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 24.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 86.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 6.4 MB/s eta 0:00:00


In [5]:
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import TranscriptsDisabled, NoTranscriptFound
import re
import yt_dlp
import whisper

def parse_video_id(url):
    patterns = [
        r'(?:v=|\/)([0-9A-Za-z_-]{11})',
        r'youtu\.be\/([0-9A-Za-z_-]{11})'
    ]
    for pattern in patterns:
        match = re.search(pattern, url)
        if match:
            return match.group(1)
    raise ValueError("Could not parse video ID from URL")

def get_transcript_native(video_id):
    ytt = YouTubeTranscriptApi()
    transcript_list = ytt.fetch(video_id)
    segments = [{'text': s.text, 'start': s.start, 'duration': s.duration} for s in transcript_list]
    full_text = " ".join([s['text'] for s in segments])
    return {
        "method": "native_captions",
        "segments": segments,
        "full_text": full_text
    }

def get_transcript_whisper(url, model_size="base"):
    print("No captions found — falling back to Whisper (this may take a minute)...")

    # Download audio only
    ydl_opts = {
        'format': 'bestaudio/best',
        'outtmpl': '/tmp/audio.%(ext)s',
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'mp3',
        }],
        'quiet': True
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([url])

    model = whisper.load_model(model_size)
    result = model.transcribe("/tmp/audio.mp3")

    return {
        "method": "whisper",
        "segments": result["segments"],
        "full_text": result["text"]
    }

def get_transcript(url, whisper_fallback=True, whisper_model="base"):
    video_id = parse_video_id(url)
    print(f"Video ID: {video_id}")

    try:
        print("Trying native captions...")
        result = get_transcript_native(video_id)
        print("✓ Got transcript via native captions")
        return result

    except (TranscriptsDisabled, NoTranscriptFound) as e:
        print(f"Native captions unavailable: {e}")
        if whisper_fallback:
            return get_transcript_whisper(url, whisper_model)
        else:
            raise

In [8]:
url = "https://www.youtube.com/watch?v=zB1lpGNWVtg"  # swap in any URL

result = get_transcript(url, whisper_fallback=True)

print(f"\nMethod used: {result['method']}")
print(f"\n--- TRANSCRIPT ---\n")
print(result['full_text'])

Video ID: zB1lpGNWVtg
Trying native captions...
✓ Got transcript via native captions

Method used: native_captions

--- TRANSCRIPT ---

We got I'm John Msad in the building. Welcome to the show. >> It is the three billion dollar darling in the space. >> This is the first AI thing that has been a mind-blowing moment for me. >> Go make some apps. >> Can you walk me through the story of this week, especially the last 24 hours? um feeling like uncertain, you know, that the the concern is is on the AI side cuz um since you guys last been here, we they literally rewrote the agent. They changed the entire architecture, the whole thing. We're all very pumped because we saw the future and that's that's awesome. Like I think we have something super special, but um going out in a broken state is is a real concern. Yeah. Yeah. But that's what happens. You go you go and travel, you come back, they rewrote the thing. It's um it's a little nerve-wracking, but I have faith in the team. >> There are tw